In [ ]:
from pathlib import Path
import cProfile
import pstats
import time

import pandas as pd
from rtnls_fundusprep.cfi_bounds import CFIBounds as Bounds

from vascx.fundus.feature_sets import *
from vascx.fundus.features.base import LayerFeature, RetinaFeature, VesselsLayerFeature
from vascx.fundus.layer import VesselTreeLayer
from vascx.fundus.loader import RetinaLoader
from vascx.fundus.retina import Retina
from vascx.fundus.vessels_layer import FundusVesselsLayer
from vascx.shared.features import FeatureSet
from vascx.utils.analysis import extract_one


In [ ]:
DATASET_CANDIDATES = [
    Path("../../samples/fundus"),
    Path("../samples/fundus"),
    Path("samples/fundus"),
]
DATASET_PATH = next(path for path in DATASET_CANDIDATES if path.exists())
FEATURE_SET_NAME = "full_v3"
PROFILE_OUT = Path("profile_extract_one.prof")
TOP_N = 40

loader = RetinaLoader.from_folder(DATASET_PATH)
example = loader.to_dict()[0].copy()
example


Folder av_path: 6 files
Folder disc_path: 6 files
Folder fundus_path: 6 files
Folder vessels_path: 6 files


{'av_path': PosixPath('../../samples/fundus/av/CHASEDB1_08L.png'),
 'disc_path': PosixPath('../../samples/fundus/discs/CHASEDB1_08L.png'),
 'fundus_path': PosixPath('../../samples/fundus/rgb/CHASEDB1_08L.png'),
 'vessels_path': PosixPath('../../samples/fundus/vessels/CHASEDB1_08L.png'),
 'id': 'CHASEDB1_08L',
 'fovea_location': (np.float64(953.4), np.float64(553.8)),
 'bounds': {'hw': (960, 999),
  'center': (np.float64(493.15861300652193), np.float64(478.5875670375119)),
  'radius': np.float64(445.84881040306396),
  'lines': {}}}

In [ ]:
profiler = cProfile.Profile()
profiler.enable()
dg =loader[0].arteries.digraph
dg =loader[1].arteries.digraph
dg =loader[2].arteries.digraph
dg =loader[3].arteries.digraph
dg =loader[4].arteries.digraph
profiler.disable()

In [ ]:
stats = pstats.Stats(profiler).strip_dirs().sort_stats("cumulative")
stats.print_stats(TOP_N)

         4556962 function calls (4331815 primitive calls) in 3.788 seconds

   Ordered by: cumulative time
   List reduced from 3815 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    3.787    1.893 interactiveshell.py:3424(run_code)
    121/2    0.002    0.000    3.787    1.893 {built-in method builtins.exec}
        1    0.000    0.000    3.786    3.786 328789822.py:1(<module>)
   1370/1    0.002    0.000    3.679    3.679 functools.py:982(__get__)
        1    0.000    0.000    3.679    3.679 layer.py:126(digraph)
        1    0.001    0.001    3.656    3.656 layer.py:80(graph)
        1    0.000    0.000    3.613    3.613 graph.py:16(make_graph)
        1    0.004    0.004    3.612    3.612 sknw.py:129(build_sknw)
        2    0.000    0.000    3.603    1.802 dispatcher.py:344(_compile_for_args)
     53/2    0.001    0.000    3.586    1.793 dispatcher.py:862(compile)
     14/2    0.000    0.000    3.

In [3]:
profiler = cProfile.Profile()
profiler.enable()
features, warning_messages = extract_one(example.copy(), FEATURE_SET_NAME)
profiler.disable()

print(f"Computed {len(features)} features")
warning_messages[:10]


Computed 67 features


[]

In [4]:
stats = pstats.Stats(profiler).strip_dirs().sort_stats("cumulative")
stats.print_stats(TOP_N)


         7657510 function calls (7430371 primitive calls) in 4.183 seconds

   Ordered by: cumulative time
   List reduced from 4216 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    4.185    2.093 interactiveshell.py:3424(run_code)
    121/2    0.001    0.000    4.185    2.093 {built-in method builtins.exec}
        1    0.000    0.000    4.185    4.185 742600487.py:1(<module>)
        1    0.000    0.000    4.142    4.142 analysis.py:15(extract_one)
        1    0.000    0.000    4.142    4.142 retina.py:88(calc_features)
 2243/276    0.002    0.000    2.429    0.009 functools.py:982(__get__)
        6    0.000    0.000    2.042    0.340 bifurcation_angles.py:58(compute)
        6    0.000    0.000    2.030    0.338 bifurcation_angles.py:50(_get_bifurcation_points)
        6    0.000    0.000    2.020    0.337 layer.py:205(filter_bifurcations)
        2    0.000    0.000    2.020    1.010 layer.py:198

In [ ]:
def profile_feature_times(ex: dict, feature_set_name: str) -> pd.DataFrame:
    """Time each feature computation on one example."""
    feature_set = FeatureSet.get_by_name(feature_set_name)
    if feature_set is None:
        raise ValueError(f"Feature set '{feature_set_name}' not found.")

    ex = ex.copy()
    if "bounds" in ex:
        bounds = ex["bounds"]
        if isinstance(bounds, dict):
            bounds = Bounds(**bounds)
        M = bounds.get_cropping_transform(1024)
        ex["bounds"] = bounds.warp(M)

    retina = Retina.from_file(**ex)
    rows = []

    for feature in feature_set:
        if isinstance(feature, RetinaFeature):
            targets = [("retina", retina)]
        elif isinstance(feature, LayerFeature):
            targets = [(layer.name, layer) for layer in retina.get_layers(VesselTreeLayer)]
        elif isinstance(feature, VesselsLayerFeature):
            targets = [(layer.name, layer) for layer in retina.get_layers(FundusVesselsLayer)]
        else:
            continue

        for target_name, target in targets:
            t0 = time.perf_counter()
            feature.compute(target)
            rows.append(
                {
                    "feature": feature.__class__.__name__,
                    "target": target_name,
                    "canonical_name": feature.canonical_name(layer_name=target_name),
                    "seconds": time.perf_counter() - t0,
                }
            )

    return pd.DataFrame(rows).sort_values("seconds", ascending=False)


profile_feature_times(example, FEATURE_SET_NAME).head(15)


In [ ]:
profiler.dump_stats(PROFILE_OUT)
PROFILE_OUT.resolve()
